# DGT PORTFOLIO

## INGENIERÍA DE DATOS

### ETL 

#### EXTRACCIÓN

In [ ]:
# Imports y configuración
import pandas as pd
import glob
import re
from pathlib import Path

DIR_XLSX    = Path("C:/Users/vinis/Desktop/VSC Codigos/Python/DGT_PRJCT/EXCELS")
OUTPUT_PATH = Path("C:/Users/vinis/Desktop/VSC Codigos/Python/DGT_PRJCT/accidentes_consolidado.csv")

COLS_VMP = ["TOT_VMP_MU24H", "TOT_VMP_MU30DF"]
ANIO_VMP = 2022

In [8]:
# Celda 2 — Convertir xlsx a csv (si aún no lo has hecho)
for archivo in sorted(DIR_XLSX.glob("*.xlsx")):
    ruta_csv = archivo.with_suffix(".csv")
    if not ruta_csv.exists():
        df = pd.read_excel(archivo)
        df.to_csv(ruta_csv, index=False, encoding="utf-8")
        print(f"Convertido: {archivo.name}")
    else:
        print(f"Ya existe: {archivo.name}")

Ya existe: Tabla-Accidentes-2016.xlsx
Ya existe: Tabla-Accidentes-2017.xlsx
Ya existe: Tabla-Accidentes-2018.xlsx
Ya existe: Tabla-Accidentes-2019.xlsx
Ya existe: TABLA_ACCIDENTES_20.xlsx
Ya existe: TABLA_ACCIDENTES_21.xlsx
Ya existe: TABLA_ACCIDENTES_22.xlsx


In [ ]:
# Detección de separador
def detectar_separador(archivo):
    with open(archivo, 'r', encoding='utf-8') as f:
        primera_linea = f.readline()
        return ";" if ";" in primera_linea else ","

archivos_csv = sorted(DIR_XLSX.glob("*.csv"))

for archivo in archivos_csv:
    print(archivo.name, "→", detectar_separador(archivo))

Tabla-Accidentes-2016.csv → ,
Tabla-Accidentes-2017.csv → ,
Tabla-Accidentes-2018.csv → ,
Tabla-Accidentes-2019.csv → ,
Tabla-Accidentes-2020.csv → ,
Tabla-Accidentes-2021.csv → ,
Tabla-Accidentes-2022.csv → ,
Tabla-Accidentes-2023.csv → ;
Tabla-Accidentes-2024.csv → ;


In [ ]:
# Consolidar
def extraer_anio(nombre):
    matches = re.findall(r"(20\d{2})", nombre)
    if matches:
        return int(matches[0])
    raise ValueError(f"No se pudo extraer el año de: {nombre}")

lista_dfs = []

for archivo in archivos_csv:
    anio = extraer_anio(archivo.name)
    sep  = detectar_separador(archivo)

    df = pd.read_csv(archivo, sep=sep, encoding="utf-8", low_memory=False)
    df.columns = df.columns.str.strip().str.upper()  # ← descomento esto

    # Homogeneizar VMP
    if anio < ANIO_VMP:
        for col in COLS_VMP:
            if col not in df.columns:
                df[col] = 0

    # ID único global
    df["ID_ACCIDENTE_GLOBAL"] = str(anio) + "_" + (pd.RangeIndex(1, len(df) + 1)).astype(str)

    print(f"[{anio}] {len(df):,} filas | sep='{sep}'")
    lista_dfs.append(df)

df_total = pd.concat(lista_dfs, ignore_index=True, sort=False)

# ID global como primera columna
cols = ["ID_ACCIDENTE_GLOBAL"] + [c for c in df_total.columns if c != "ID_ACCIDENTE_GLOBAL"]
df_total = df_total[cols]

print(f"\nTotal filas: {len(df_total):,} | Total columnas: {len(df_total.columns)}")

[2016] 102,362 filas | sep=','
[2017] 102,233 filas | sep=','
[2018] 102,299 filas | sep=','
[2019] 104,080 filas | sep=','
[2020] 72,959 filas | sep=','
[2021] 89,862 filas | sep=','
[2022] 97,916 filas | sep=','
[2023] 101,306 filas | sep=';'
[2024] 101,996 filas | sep=';'

Total filas: 875,013 | Total columnas: 76


In [17]:
df_total.to_csv(OUTPUT_PATH, index=False, sep=";", encoding="utf-8")
print(f"✅ Guardado en: {OUTPUT_PATH}")

✅ Guardado en: C:\Users\vinis\Desktop\VSC Codigos\Python\DGT_PRJCT\accidentes_consolidado.csv


#### TRANSFORMAR

In [1]:
import pandas as pd

df_dgt= pd.read_csv('ACCIDENTES_TOTALES.csv', sep=',', encoding= 'utf-8')
df_dgt

C:\Users\vinis\AppData\Local\Temp\ipykernel_29704\1826448744.py:3: DtypeWarning: Columns (11,49) have mixed types. Specify dtype option on import or set low_memory=False.
  df_dgt= pd.read_csv('ACCIDENTES_TOTALES.csv', sep=',', encoding= 'utf-8')


,SECUENCIAL,ANYO,MES,DIA_SEMANA,HORA,COD_PROVINCIA,COD_MUNICIPIO,ISLA,ZONA,ZONA_AGRUPADA,...,CONDICION_ILUMINACION,CONDICION_METEO,CONDICION_NIEBLA,CONDICION_VIENTO,VISIB_RESTRINGIDA_POR,ACERA,TRAZADO_PLANTA,TOT_VMP_MU24H,TOT_VMP_MU30DF,ID_ACCIDENTE
0,1.0,2016,3,4,16,1,1059.0,NaN,3,2,...,1,7,NaN,NaN,18,998,998,NaN,NaN,NaN
1,2.0,2016,3,4,14,1,1059.0,NaN,3,2,...,1,7,NaN,NaN,18,999,998,NaN,NaN,NaN
2,3.0,2016,3,4,7,1,1059.0,NaN,3,2,...,4,7,NaN,NaN,18,998,998,NaN,NaN,NaN
3,4.0,2016,3,3,18,1,1059.0,NaN,3,2,...,1,7,NaN,NaN,18,998,998,NaN,NaN,NaN
4,5.0,2016,3,2,20,1,1059.0,NaN,3,2,...,4,7,NaN,NaN,18,998,998,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
875008,NaN,2024,10,1,14,51,51001.0,NaN,3,2,...,1,1,NaN,NaN,9,998,998,NaN,0.0,101992.0
875009,NaN,2024,11,6,10,51,51001.0,NaN,3,2,...,1,1,NaN,NaN,1,998,998,NaN,0.0,101993.0
875010,NaN,2024,12,5,8,51,51001.0,NaN,3,2,...,1,1,NaN,NaN,1,998,998,NaN,0.0,101994.0
875011,NaN,2024,12,2,18,51,51001.0,NaN,3,2,...,1,1,NaN,NaN,1,998,998,NaN,0.0,101995.0


In [4]:
df_dgt[[ 'ANYO', 'ID_ACCIDENTE']]

,ANYO,ID_ACCIDENTE
0,2016,NaN
1,2016,NaN
2,2016,NaN
3,2016,NaN
4,2016,NaN
...,...,...
875008,2024,101992.0
875009,2024,101993.0
875010,2024,101994.0
875011,2024,101995.0


In [3]:
df_dgt.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 875013 entries, 0 to 875012
Data columns (total 75 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   SECUENCIAL                483933 non-null  float64
 1   ANYO                      875013 non-null  int64  
 2   MES                       875013 non-null  int64  
 3   DIA_SEMANA                875013 non-null  int64  
 4   HORA                      875013 non-null  int64  
 5   COD_PROVINCIA             875013 non-null  int64  
 6   COD_MUNICIPIO             872805 non-null  float64
 7   ISLA                      51411 non-null   float64
 8   ZONA                      875013 non-null  int64  
 9   ZONA_AGRUPADA             875013 non-null  int64  
 10  CARRETERA                 875012 non-null  object 
 11  KM                        375284 non-null  object 
 12  SENTIDO_1F                875013 non-null  int64  
 13  TITULARIDAD_VIA           875013 non-null  i

In [4]:
#Buscar valores nulos en filas
consulta= df_dgt[df_dgt['TIPO_VIA'].isnull()]
consulta[['ANYO','TIPO_VIA', 'TITULARIDAD_VIA','COD_PROVINCIA','COD_MUNICIPIO','KM','CARRETERA', 'TIPO_VIA']]

,ANYO,TIPO_VIA,TITULARIDAD_VIA,COD_PROVINCIA,COD_MUNICIPIO,KM,CARRETERA,TIPO_VIA
158053,2017,NaN,4,28,28079.0,NaN,No inventariada,NaN
158299,2017,NaN,4,28,28079.0,NaN,No inventariada,NaN
158602,2017,NaN,4,28,28079.0,NaN,No inventariada,NaN
159254,2017,NaN,4,28,28079.0,NaN,No inventariada,NaN
159427,2017,NaN,4,28,28079.0,NaN,No inventariada,NaN
...,...,...,...,...,...,...,...,...
266640,2018,NaN,4,28,28079.0,NaN,No inventariada,NaN
267050,2018,NaN,4,28,28079.0,NaN,No inventariada,NaN
267382,2018,NaN,4,28,28079.0,NaN,No inventariada,NaN
267909,2018,NaN,4,28,28079.0,NaN,No inventariada,NaN


In [5]:
df_dgt['COD_MUNICIPIO'].value_counts().sort_index(ascending=True)

COD_MUNICIPIO
0.0        105846
1002.0        148
1036.0         70
1051.0        106
1059.0       5687
            ...  
50272.0       183
50297.0      9509
50298.0       169
51001.0      2716
52001.0      3485
Name: count, Length: 1350, dtype: int64

In [6]:
df_dgt['COD_MUNICIPIO'] = df_dgt['COD_MUNICIPIO'].fillna(0)
df_dgt['ISLA'] = df_dgt['ISLA'].fillna(0)
df_dgt['NUDO_INFO']= df_dgt['NUDO_INFO'].fillna(999)
df_dgt['CONDICION_NIEBLA']= df_dgt['CONDICION_NIEBLA'].fillna(0)
df_dgt['CONDICION_VIENTO']= df_dgt['CONDICION_VIENTO'].fillna(0)
df_dgt['TOT_VMP_MU24H']= df_dgt['TOT_VMP_MU24H'].fillna(0)
df_dgt['TOT_VMP_MU30DF'] = df_dgt['TOT_VMP_MU30DF'].fillna(0)
df_dgt['KM']= df_dgt['KM'].fillna('Desconocido')
df_dgt['CARRETERA']= df_dgt['CARRETERA'].fillna('No inventariada')
df_dgt['NUDO']= df_dgt['NUDO'].fillna(2)
df_dgt['CARRETERA_CRUCE']= df_dgt['CARRETERA_CRUCE'].fillna('No inventariada')
df_dgt['TIPO_VIA']= df_dgt['TIPO_VIA'].fillna(14)
df_dgt['TIPO_ACCIDENTE']= df_dgt['TIPO_ACCIDENTE'].fillna(20)

In [7]:
df_dgt['CARRETERA_CRUCE'].value_counts()

CARRETERA_CRUCE
No inventariada    851141
A-7                   507
M-50                  289
N-340                 284
A-4                   248
                    ...  
VP-3014                 1
VA-912                  1
CV SIMPLÓN              1
VP-5001                 1
CV-473                  1
Name: count, Length: 5001, dtype: int64

In [8]:
df_dgt.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 875013 entries, 0 to 875012
Data columns (total 75 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   SECUENCIAL                483933 non-null  float64
 1   ANYO                      875013 non-null  int64  
 2   MES                       875013 non-null  int64  
 3   DIA_SEMANA                875013 non-null  int64  
 4   HORA                      875013 non-null  int64  
 5   COD_PROVINCIA             875013 non-null  int64  
 6   COD_MUNICIPIO             875013 non-null  float64
 7   ISLA                      875013 non-null  float64
 8   ZONA                      875013 non-null  int64  
 9   ZONA_AGRUPADA             875013 non-null  int64  
 10  CARRETERA                 875013 non-null  object 
 11  KM                        875013 non-null  object 
 12  SENTIDO_1F                875013 non-null  int64  
 13  TITULARIDAD_VIA           875013 non-null  i

In [9]:
df_dgt.ANYO.value_counts().sort_index(ascending=False)

ANYO
2024    101996
2023    101306
2022     97916
2021     89862
2020     72959
2019    104080
2018    102299
2017    102233
2016    102362
Name: count, dtype: int64

In [10]:
df_dgt['TOTAL_MU30DF'].value_counts()

TOTAL_MU30DF
0     860680
1      13459
2        708
3        127
4         24
5         10
7          2
6          2
13         1
Name: count, dtype: int64

**Feature Engineering**

In [11]:
df_dgt['ACC_MORTAL']= (df_dgt['TOTAL_MU30DF'] > 0).astype(int)
df_dgt['ACC_GRAVES']= (df_dgt['TOTAL_HG30DF'] > 0).astype(int)

Se crea una variable binaria de mortalidad centrándonos en las muertes dentro de un mes. Esto se debe a que las muertes se pueden producir en varios días.

Lo mismo pasa con los heridos graves

In [12]:
df_dgt['HORA'].value_counts().sort_index(ascending=True)

HORA
0     14067
1      9532
2      6764
3      5815
4      6016
5      8856
6     15823
7     31095
8     43707
9     46346
10    44028
11    51182
12    56247
13    61967
14    65710
15    53550
16    48591
17    52192
18    56863
19    56464
20    50143
21    39732
22    30753
23    19570
Name: count, dtype: int64

In [13]:
def clasificar_franja(hora):
    if 0 <= hora < 6:
        return "Madrugada"
    elif 6 <= hora < 12:
        return "Mañana"
    elif 12 <= hora < 20:
        return "Tarde"
    else:
        return "Noche"

df_dgt["FRANJA_HORARIA"] = df_dgt["HORA"].apply(clasificar_franja)


In [14]:
def estacion_año(mes):
    if mes in [12, 1, 2]:
        return "Invierno"
    elif mes in [3, 4, 5]:
        return "Primavera"
    elif mes in [6, 7, 8]:
        return "Verano"
    else:
        return "Otoño"

df_dgt['ESTACION']= df_dgt['MES'].apply(estacion_año)

In [15]:
df_dgt.columns

Index(['SECUENCIAL', 'ANYO', 'MES', 'DIA_SEMANA', 'HORA', 'COD_PROVINCIA',
       'COD_MUNICIPIO', 'ISLA', 'ZONA', 'ZONA_AGRUPADA', 'CARRETERA', 'KM',
       'SENTIDO_1F', 'TITULARIDAD_VIA', 'TIPO_VIA', 'TIPO_ACCIDENTE',
       'TOTAL_MU24H', 'TOTAL_HG24H', 'TOTAL_HL24H', 'TOTAL_VICTIMAS_24H',
       'TOTAL_MU30DF', 'TOTAL_HG30DF', 'TOTAL_HL30DF', 'TOTAL_VICTIMAS_30DF',
       'TOTAL_VEHICULOS', 'TOT_PEAT_MU24H', 'TOT_BICI_MU24H',
       'TOT_CICLO_MU24H', 'TOT_MOTO_MU24H', 'TOT_TUR_MU24H', 'TOT_FURG_MU24H',
       'TOT_CAM_MENOS3500_MU24H', 'TOT_CAM_MAS3500_MU24H', 'TOT_BUS_MU24H',
       'TOT_OTRO_MU24H', 'TOT_SINESPECIF_MU24H', 'TOT_PEAT_MU30DF',
       'TOT_BICI_MU30DF', 'TOT_CICLO_MU30DF', 'TOT_MOTO_MU30DF',
       'TOT_TUR_MU30DF', 'TOT_FURG_MU30DF', 'TOT_CAM_MENOS3500_MU30DF',
       'TOT_CAM_MAS3500_MU30DF', 'TOT_BUS_MU30DF', 'TOT_OTRO_MU30DF',
       'TOT_SINESPECIF_MU30DF', 'NUDO', 'NUDO_INFO', 'CARRETERA_CRUCE',
       'PRIORI_NORMA', 'PRIORI_AGENTE', 'PRIORI_SEMAFORO', 'PRI

In [16]:
df_dgt['TOTAL_HG30DF'].value_counts()

TOTAL_HG30DF
0     803801
1      65344
2       4796
3        746
4        227
5         70
6         18
7          7
9          2
23         1
12         1
Name: count, dtype: int64